# 07 · Voice in, voice out
**Time:** 20 min · **Goal:** turn a spoken question into an answer and hear it spoken back.

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

> Audio models and voices change often. Check https://ai.google.dev/gemini-api/docs/audio and https://ai.google.dev/gemini-api/docs/speech-generation if a cell fails. For real-time conversation, use the Live API workshop: https://github.com/saurabh1611a7458-pixel/Agent-Voice

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()

## Step 1 · Upload a short recording
Record a 10 to 20 second question on your phone in your language, then upload it (wav, mp3 or m4a).

In [ ]:
from google.colab import files
up = files.upload()
fname = list(up.keys())[0]
audio = up[fname]
mime = "audio/wav" if fname.endswith(".wav") else "audio/mp3" if fname.endswith(".mp3") else "audio/mp4"
print(fname, len(audio), "bytes")

## Step 2 · Transcribe and answer

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents=[types.Part.from_bytes(data=audio, mime_type=mime),
              "1) Transcribe this audio exactly. 2) Then answer the farmer's question briefly in the same language."])
print(r.text)

## Step 3 · Speak the answer back (text to speech)

In [ ]:
import wave
from IPython.display import Audio
answer_text = "Pole. Angalia mifereji ya maji na uwasiliane na afisa ugani."   # replace with your agent's answer
TTS_MODELS = ["gemini-3.1-flash-tts-preview", "gemini-2.5-flash-preview-tts"]   # first ID is unverified, second is in Google's docs
tts = None
for tm in TTS_MODELS:
    try:
        tts = client.models.generate_content(
            model=tm, contents=answer_text,
            config=types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=types.SpeechConfig(voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")))))
        print("TTS model:", tm); break
    except Exception as e:
        print("Skipping", tm, "->", str(e)[:100])
if tts is None:
    raise RuntimeError("No TTS model worked. Copy a current TTS model name from AI Studio into TTS_MODELS.")
pcm = tts.candidates[0].content.parts[0].inline_data.data
with wave.open("answer.wav", "wb") as f:
    f.setnchannels(1); f.setsampwidth(2); f.setframerate(24000); f.writeframes(pcm)
Audio("answer.wav")

## Your turn
- Does the voice pronounce your language well? Rate it 1 to 5 for the report card. Speech output in Swahili is listed for Google Cloud's Gemini-TTS (preview); whether the Gemini API TTS model speaks it well is something to test, not assume.
- Compare with Live Translate: https://ai.google.dev/gemini-api/docs/live-api/live-translate